In [14]:

import csv
import qrcode
import random
import shutil
from pathlib import Path


In [15]:

CHARS = "ABCDEFGHJKLMNPQRSTUVWXYZabcdefghjkmnpqrstuvwxyz23456789"
## 去除了0 1 O I L


In [17]:

def generate_code(groups=4, group_len=4):
    parts = []
    for _ in range(groups):
        part = ''.join(random.choice(CHARS) for _ in range(group_len))
        parts.append(part)
    return "-".join(parts)


count = 300
csv_file = Path("ticket_codes.csv")
output_dir = Path("qrcodes")


In [22]:

# 1. 删除旧 CSV
if csv_file.exists():
    csv_file.unlink()

# 2. 删除旧二维码文件夹及其中100张图
if output_dir.exists():
    shutil.rmtree(output_dir)

# 3. 重新创建二维码文件夹
output_dir.mkdir(exist_ok=True)

# 4. 重新生成100个兑换码
codes = set()

while len(codes) < count:
    codes.add(generate_code(groups=4, group_len=4))
    # groups=4 表示4组，group_len=4 表示每组4位
    # 例如：K7QW-9M2X-TR6P-B8ND

# 5. 重新生成新的 CSV 和新的二维码
with open(csv_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["ticket_no", "code", "qr_file"])

    for i, code in enumerate(sorted(codes), start=1):
        ticket_no = f"A{i:04d}"

        qr = qrcode.QRCode(
            version=None,
            error_correction=qrcode.constants.ERROR_CORRECT_H,
            box_size=12,
            border=4
            )
        
        qr.add_data(code)
        qr.make(fit=True)

        img = qr.make_image(fill_color="black", back_color="white")
        filename = output_dir / f"{ticket_no}_{code}.png"
        img.save(filename)

        writer.writerow([ticket_no, code, str(filename)])